# EM Data Overview

Overview of electron microscopy datasets in `/myhome/data/sdate/shared/em_data`.

For each subfolder, up to 4 volumes are shown with:
- **Top row**: real-space center slices along XY, XZ, YZ axes
- **Bottom row**: `log1p(|FFT|)` center slices along the same axes — reveals the missing wedge as a low-amplitude region in Fourier space

In [1]:
# !pip install mrcfile


In [2]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ── project path ──────────────────────────────────────────────────────────────
sys.path.insert(0, '/myhome/sdate')

import importlib
import ladiff.datasets.em_io as _em_mod
importlib.reload(_em_mod)

import torch
import lovely_tensors as lt
lt.monkey_patch()

from ladiff.datasets.em_io import (
    load_mrc_volume,
    collect_em_files,
)

plt.rcParams['figure.dpi'] = 100

DATA_ROOT = Path('/myhome/data/sdate/shared/em_data')
OUT_ROOT  = Path('/myhome/data/sdate/shared/em_data_npy')   # where .npy files are saved 


zarr not installed


In [3]:
def center_slice(vol: np.ndarray, axis: int) -> np.ndarray:
    """Return the central 2-D slice of *vol* perpendicular to *axis*."""
    return np.take(vol, vol.shape[axis] // 2, axis=axis)


def fft_log_magnitude(vol: np.ndarray) -> np.ndarray:
    """Return ``log1p(|3-D FFT|)`` of *vol*, fft-shifted to DC-centre."""
    f = np.fft.fftshift(np.fft.fftn(vol))
    return np.log1p(np.abs(f))


def visualize_volume(vol: np.ndarray, title: str) -> None:
    """Plot centre slices (real space + FFT log-magnitude) along all three axes."""
    fft_vol = fft_log_magnitude(vol)

    axes_labels = ['XY (⊥Z)', 'XZ (⊥Y)', 'YZ (⊥X)']
    ax_indices  = [0, 1, 2]   # axis to slice along: Z=0, Y=1, X=2

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    fig.suptitle(title, fontsize=11, y=1.01)

    for row, (data, rl) in enumerate(zip([vol, fft_vol], ['Real space', 'log1p(|FFT|)'])):
        for col, (axis_idx, label) in enumerate(zip(ax_indices, axes_labels)):
            sl = center_slice(data, axis_idx)
            vmin, vmax = np.percentile(sl, [1, 99])
            axes[row, col].imshow(sl, cmap='gray', vmin=vmin, vmax=vmax, origin='lower')
            axes[row, col].set_title(f'{rl} — {label}', fontsize=8)
            axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()


# ── Collect all .mrc / .rec files, up to 4 per subfolder ─────────────────────
MAX_PER_FOLDER = 4

folders = collect_em_files(DATA_ROOT, max_per_folder=MAX_PER_FOLDER)
for folder, files in folders.items():
    print(f"{folder.relative_to(DATA_ROOT)}:  {len(files)} file(s) shown")
    for f in files:
        print(f"   {f.name}")


HIV/demo_data/tomograms:  3 file(s) shown
   TS01-wbp.rec
   TS43-wbp.rec
   TS45-wbp.rec
HIV/expected_output:  1 file(s) shown
   TS01_corrected.mrc
neuron:  2 file(s) shown
   synapse-bin4-5i-demo.rec
   synapse-bin4-5i-demo_corrected.mrc


---
## Neuron dataset

In [ ]:
folder_key = DATA_ROOT / 'neuron'
files = folders.get(folder_key, [])
print(f"Folder: {folder_key.relative_to(DATA_ROOT)}  — {len(files)} files")

for fpath in files:
    vol = load_mrc_volume(fpath)
    print(f"\n{fpath.name}  →  shape (Z,Y,X) = {vol.shape},  dtype = {vol.dtype},  "
          f"min/max = {vol.min():.3g} / {vol.max():.3g}")
    visualize_volume(vol, title=fpath.name)


---
## HIV dataset — demo tomograms

In [ ]:
folder_key = DATA_ROOT / 'HIV' / 'demo_data' / 'tomograms'
files = folders.get(folder_key, [])
print(f"Folder: {folder_key.relative_to(DATA_ROOT)}  — {len(files)} files")

for fpath in files:
    vol = load_mrc_volume(fpath)
    print(f"\n{fpath.name}  →  shape (Z,Y,X) = {vol.shape},  dtype = {vol.dtype},  "
          f"min/max = {vol.min():.3g} / {vol.max():.3g}")
    visualize_volume(vol, title=fpath.name)


---
## HIV dataset — expected output (IsoNet-corrected)

In [ ]:
folder_key = DATA_ROOT / 'HIV' / 'expected_output'
files = folders.get(folder_key, [])
print(f"Folder: {folder_key.relative_to(DATA_ROOT)}  — {len(files)} files")

for fpath in files:
    vol = load_mrc_volume(fpath)
    print(f"\n{fpath.name}  →  shape (Z,Y,X) = {vol.shape},  dtype = {vol.dtype},  "
          f"min/max = {vol.min():.3g} / {vol.max():.3g}")
    visualize_volume(vol, title=fpath.name)


---
## Save EM Volumes as NPY

Saves all collected EM volumes to `OUT_ROOT` as normalised float32 ``.npy`` files,
with a companion ``_norm.json`` sidecar (compatible with ``NpyVolumeSliceDataset``).

**Normalisation**: 2nd–98th percentile → ``[0, 1]``.

**Axis permutation** (`permute=True`): the raw EM volumes have the tilt axis along **Y** (axis 1),
so the missing wedge is visible in the **XZ Fourier slice** (⊥Y).
After `permute_em_to_tilt_axis0`, axes are reordered as `(Y, Z, X)`, placing the tilt axis at
position 0 — the same convention used in the Fourier-cone removal pipeline (`TILT_AXIS=0`).
The missing wedge is then visible in the **XY Fourier slice** (⊥Z, axis 0) of the saved volume.


In [4]:
import importlib
import ladiff.datasets.em_io as _em_mod
importlib.reload(_em_mod)

from ladiff.datasets.em_io import save_em_volume_npy, load_mrc_volume, collect_em_files

# ── Config ────────────────────────────────────────────────────────────────────
P_LOW, P_HIGH = 2.0, 98.0   # percentile bounds for [0, 1] normalisation
PERMUTE       = True          # permute (Z, Y, X) → (Y, Z, X) to match TILT_AXIS=0
MAKE_CUBIC    = True          # center-crop to a cube (side = min dimension) before saving
MIN_SIZE      = 256           # if any dim < MIN_SIZE after cropping, rescale to MIN_SIZE^3

OUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Output root    : {OUT_ROOT}")
print(f"Normalisation  : p{P_LOW:.0f} → 0,  p{P_HIGH:.0f} → 1")
print(f"Axis permute   : {PERMUTE}  (Y→Z so missing wedge aligns with TILT_AXIS=0)")
print(f"Make cubic     : {MAKE_CUBIC}  (center-crop to min-dimension cube)")
print(f"Min size       : {MIN_SIZE}  (rescale to {MIN_SIZE}³ if any dim is smaller)")


Output root    : /myhome/data/sdate/shared/em_data_npy
Normalisation  : p2 → 0,  p98 → 1
Axis permute   : True  (Y→Z so missing wedge aligns with TILT_AXIS=0)
Make cubic     : True  (center-crop to min-dimension cube)
Min size       : 256  (rescale to 256³ if any dim is smaller)


In [5]:
# Collect ALL .mrc/.rec files (no per-folder cap) for saving
from collections import defaultdict

def _collect_all_em_files(root: Path):
    groups = defaultdict(list)
    for f in sorted(root.rglob('*')):
        if f.is_file() and f.suffix.lower() in ('.mrc', '.rec'):
            groups[f.parent].append(f)
    return dict(sorted(groups.items()))

all_folders = _collect_all_em_files(DATA_ROOT)
total_files = sum(len(v) for v in all_folders.values())
print(f"Found {total_files} volumes across {len(all_folders)} subfolder(s)\n")

all_folders

Found 6 volumes across 3 subfolder(s)



{Path('/myhome/data/sdate/shared/em_data/HIV/demo_data/tomograms'): [Path('/myhome/data/sdate/shared/em_data/HIV/demo_data/tomograms/TS01-wbp.rec'),
  Path('/myhome/data/sdate/shared/em_data/HIV/demo_data/tomograms/TS43-wbp.rec'),
  Path('/myhome/data/sdate/shared/em_data/HIV/demo_data/tomograms/TS45-wbp.rec')],
 Path('/myhome/data/sdate/shared/em_data/HIV/expected_output'): [Path('/myhome/data/sdate/shared/em_data/HIV/expected_output/TS01_corrected.mrc')],
 Path('/myhome/data/sdate/shared/em_data/neuron'): [Path('/myhome/data/sdate/shared/em_data/neuron/synapse-bin4-5i-demo.rec'),
  Path('/myhome/data/sdate/shared/em_data/neuron/synapse-bin4-5i-demo_corrected.mrc')]}

In [6]:
saved_records = []  # list of dicts for the summary table

for folder, files in all_folders.items():
    rel = folder.relative_to(DATA_ROOT)
    out_dir = OUT_ROOT / rel
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"── {rel}  ({len(files)} file(s))")

    for fpath in files:
        vol = load_mrc_volume(fpath)
        out_npy = out_dir / (fpath.stem + '.npy')
        results = save_em_volume_npy(
            vol, out_npy, p_low=P_LOW, p_high=P_HIGH,
            permute=PERMUTE, make_cubic=MAKE_CUBIC, min_size=MIN_SIZE,
        )

        # Compute reported saved shape for display
        perm_axes = [1, 0, 2] if PERMUTE else [0, 1, 2]
        perm_shape = tuple(vol.shape[i] for i in perm_axes)
        if MAKE_CUBIC:
            s = min(perm_shape)
            saved_shape = (max(s, MIN_SIZE),) * 3 if MIN_SIZE > 0 else (s, s, s)
        else:
            saved_shape = (MIN_SIZE,) * 3 if (MIN_SIZE > 0 and min(perm_shape) < MIN_SIZE) else perm_shape

        n_tiles = len(results)
        print(f"   {fpath.name:<50s}  {str(vol.shape):<18s}→  {str(saved_shape):<18s}  {n_tiles} tile(s)")
        for npy_path, norm_path in results:
            saved_records.append({
                'subfolder'    : str(rel),
                'source_file'  : fpath.name,
                'original_shape': vol.shape,
                'saved_shape'  : saved_shape,
                'npy_path'     : npy_path,
                'norm_path'    : norm_path,
            })

print(f"\nDone — {len(saved_records)} tile(s) saved to {OUT_ROOT}")


── HIV/demo_data/tomograms  (3 file(s))
   TS01-wbp.rec                                        (200, 464, 480)   →  (256, 256, 256)     4 tile(s)
   TS43-wbp.rec                                        (125, 479, 463)   →  (256, 256, 256)     9 tile(s)
   TS45-wbp.rec                                        (125, 464, 480)   →  (256, 256, 256)     9 tile(s)
── HIV/expected_output  (1 file(s))
   TS01_corrected.mrc                                  (200, 464, 480)   →  (256, 256, 256)     4 tile(s)
── neuron  (2 file(s))
   synapse-bin4-5i-demo.rec                            (214, 831, 530)   →  (256, 256, 256)     6 tile(s)
   synapse-bin4-5i-demo_corrected.mrc                  (214, 831, 530)   →  (256, 256, 256)     6 tile(s)

Done — 38 tile(s) saved to /myhome/data/sdate/shared/em_data_npy


In [7]:
# ── Verify: reload one saved tile per source file and confirm range ────────────
import json

print("Verification — reloading first tile per source file:\n")
seen_sources = set()
for rec in saved_records:
    key = (rec['subfolder'], rec['source_file'])
    if key in seen_sources:
        continue
    seen_sources.add(key)

    arr = np.load(str(rec['npy_path']))
    with open(str(rec['norm_path'])) as _f:
        norm = json.load(_f)

    print(f"{rec['subfolder']} / {rec['source_file']}")
    print(f"   tile  : {rec['npy_path'].name}")
    print(f"   shape : {arr.shape}")
    print(f"   range : [{arr.min():.4f}, {arr.max():.4f}]  (expected [0, 1])")
    print(f"   norm  : min={norm['norm_min']:.4f}  max={norm['norm_max']:.4f}\n")


Verification — reloading first tile per source file:

HIV/demo_data/tomograms / TS01-wbp.rec
   tile  : TS01-wbp_tile_0_0_0.npy
   shape : (256, 256, 256)
   range : [0.0000, 1.0000]  (expected [0, 1])
   norm  : min=0.0000  max=1.0000

HIV/demo_data/tomograms / TS43-wbp.rec
   tile  : TS43-wbp_tile_0_0_0.npy
   shape : (256, 256, 256)
   range : [0.0000, 1.0000]  (expected [0, 1])
   norm  : min=0.0000  max=1.0000

HIV/demo_data/tomograms / TS45-wbp.rec
   tile  : TS45-wbp_tile_0_0_0.npy
   shape : (256, 256, 256)
   range : [0.0000, 1.0000]  (expected [0, 1])
   norm  : min=0.0000  max=1.0000

HIV/expected_output / TS01_corrected.mrc
   tile  : TS01_corrected_tile_0_0_0.npy
   shape : (256, 256, 256)
   range : [0.0000, 1.0000]  (expected [0, 1])
   norm  : min=0.0000  max=1.0000

neuron / synapse-bin4-5i-demo.rec
   tile  : synapse-bin4-5i-demo_tile_0_0_0.npy
   shape : (256, 256, 256)
   range : [0.0000, 1.0000]  (expected [0, 1])
   norm  : min=0.0000  max=1.0000

neuron / synapse